# Misspelling Corrections

Both the ground truth and the transcriptions contain several misspellings. This is because the ground truths were generated by human annotators, the transcriptions were obtained through a listener panel, and misspellings can naturally occur when typing.

To correct these misspellings, we created a class that takes a `CSV` file containing a series of spelling corrections and processes all sequences, replacing incorrect spellings with the correct forms.

First, to identify misspellings in the dataset, we compared all words from the sequences against a pronunciation dictionary. Words not found in the dictionary were classified either as obvious misspellings (e.g. `DIDNT` → `DIDN'T` or `AWNSER` → `ANSWER`) or as words for which a pronunciation needed to be generated.

## Class CheckSpellings

The class `CheckSpellings` takes a CSV file containing common misspellings and their corrections in its constructor.
The method `fix_misspellings` accepts a string sequence and returns a corrected version of the text, replacing all words that match entries in the dictionary.

```python
class CheckSpellings:
    """
    Class to check and correct common misspellings in text based on a provided dictionary.
    """
    def __init__(self, spellings_file: str):
        """
        Constructor
        
        Args:
            spellings_file (str): Path to the CSV file with common misspellings.
        """
        self.spellings = {}
        with open(spellings_file, "r") as f:
            for line in f:
                parts = line.strip().split(",", 1)
                if len(parts) == 2:
                    self.spellings[parts[0].lower()] = parts[1].lower()
                else:
                    self.spellings[parts[0].lower()] = ""

    def fix_misspellings(self, text: str) -> str:
        """
        Fix common misspellings in the text based on a provided dictionary.
        Replace whole words, even when surrounded by punctuation.
        Args:
            text (str): Input text to be corrected.
        Returns:
            str: Corrected text.
        """

        def replace_word(match):
            word = match.group(0)
            return self.spellings.get(word.lower(), word)

        # Match word-like tokens including punctuation like (), [], etc.
        # This will match (something), 'hello', etc.
        pattern = r"[^\s]+"
        corrected_text = re.sub(pattern, replace_word, text)
        return corrected_text
```

### Example for Correcting Spelling

In this example, we will correct the spelling of `useless` and `wasn't`.

In [2]:
from spellings import CheckSpellings

# Spellings object using ```spelling_corrections.csv``` file
spellings = CheckSpellings("../input_files/spelling_corrections.csv")
# The input sequence
input_sequence = "correcting words usless and wasnt."
# generate version with corrected spelling
corrected_text = spellings.fix_misspellings(input_sequence)

In [3]:
from IPython.display import display, Markdown

display(Markdown("**Original form:**"))
display(Markdown(f"- `{input_sequence}`"))
display(Markdown("**Corrected form:**"))
display(Markdown(f"- `{corrected_text}`"))

**Original form:**

- `correcting words usless and wasnt.`

**Corrected form:**

- `correcting words  useless and wasnt.`